# LSTM Stock Price Predictor — EDA Notebook
## Exploratory Data Analysis & Model Development

This notebook walks through downloading historical stock data, engineering features, and exploring model architecture before production training.

## 1. Import Required Libraries

In [ ]:
# Set seeds for reproducibility
import random
import numpy as np
import torch

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Data & ML libraries
import pandas as pd
import yfinance as yf
import pandas_ta as ta
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import logging

# PyTorch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Load and Explore Data

In [ ]:
# Download AAPL data
ticker = "AAPL"
start_date = "2018-01-01"
end_date = "2024-12-31"

logger.info(f"Downloading {ticker} data from {start_date} to {end_date}...")
df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True, progress=False)

# Drop NaN in OHLCV
df = df.dropna(subset=['Open', 'High', 'Low', 'Close', 'Volume'])

print(f"\nShape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDate range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"\nFirst 5 rows:")
print(df.head())

In [ ]:
# Plot raw close prices
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df.index, df['Close'], linewidth=1, alpha=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Close Price ($)')
ax.set_title(f'{ticker} Historical Close Price ({start_date} to {end_date})')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"\nPrice Statistics:")
print(df['Close'].describe())

## 3. Feature Engineering with pandas-ta

In [ ]:
# Add technical indicators
logger.info("Adding technical indicators...")

# Make a copy to preserve original
df_features = df.copy()

# Add indicators
df_features.ta.rsi(length=14, append=True)
df_features.ta.macd(fast=12, slow=26, signal=9, append=True)
df_features.ta.bbands(length=20, std=2.0, append=True)
df_features.ta.sma(length=20, append=True)
df_features.ta.sma(length=50, append=True)
df_features.ta.ema(length=12, append=True)

print(f"\nFeatures after adding indicators:")
print(f"Shape: {df_features.shape}")
print(f"Columns: {df_features.columns.tolist()}")
print(f"\nMissing values (first 60 rows typically have NaN from indicator warm-up):")
print(df_features.isnull().sum())

In [ ]:
# Drop rows with NaN from indicator warm-up
initial_rows = len(df_features)
df_features = df_features.dropna()
rows_dropped = initial_rows - len(df_features)

print(f"Dropped {rows_dropped} rows due to NaN")
print(f"Remaining rows: {len(df_features)}")
print(f"\nFinal features:")
print(df_features.head())

In [ ]:
# Select features to use
features = [
    "Close", "Volume",
    "RSI_14", "MACD_12_26_9", "MACDh_12_26_9",
    "BBL_20_2.0", "BBU_20_2.0",
    "SMA_20", "SMA_50", "EMA_12"
]

# Create feature matrix
df_model = df_features[features].copy()

print(f"Model features shape: {df_model.shape}")
print(f"\nFeature correlation matrix:")
corr_matrix = df_model.corr()
print(corr_matrix['Close'].sort_values(ascending=False))

In [ ]:
# Visualize correlation matrix
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
fig.tight_layout()
plt.show()

## 4. Data Preprocessing and Scaling

In [ ]:
# Fit scaler on training data only (prevent data leakage)
train_ratio = 0.70
val_ratio = 0.15

n_samples = len(df_model)
train_end = int(n_samples * train_ratio)
val_end = int(n_samples * (train_ratio + val_ratio))

# Split
df_train = df_model.iloc[:train_end]
df_val = df_model.iloc[train_end:val_end]
df_test = df_model.iloc[val_end:]

print(f"Train samples: {len(df_train)} ({train_ratio*100:.0f}%)")
print(f"Val samples: {len(df_val)} ({val_ratio*100:.0f}%)")
print(f"Test samples: {len(df_test)} ({(1-train_ratio-val_ratio)*100:.0f}%)")

# Fit scaler on training data ONLY
scaler = MinMaxScaler()
scaler.fit(df_train)

# Scale all data
scaled_train = scaler.transform(df_train)
scaled_val = scaler.transform(df_val)
scaled_test = scaler.transform(df_test)

print(f"\nScaled train data shape: {scaled_train.shape}")
print(f"Scaled train data range: [{scaled_train.min():.4f}, {scaled_train.max():.4f}]")
print(f"✓ No data leakage: scaler fit on train only")

In [ ]:
# Verify no data leakage
print("Data Leakage Check:")
print(f"Last train date: {df_train.index[-1].date()}")
print(f"First val date: {df_val.index[0].date()}")
print(f"Last val date: {df_val.index[-1].date()}")
print(f"First test date: {df_test.index[0].date()}")
print(f"Last test date: {df_test.index[-1].date()}")

# Check chronological order
assert df_train.index[-1] < df_val.index[0], "Train/Val overlap!"
assert df_val.index[-1] < df_test.index[0], "Val/Test overlap!"
print(f"\n✓ Chronological integrity verified")

## 5. Create Sequences for LSTM

In [ ]:
def create_sequences(data, lookback=60):
    """Create sliding window sequences."""
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])  # Last 60 days
        y.append(data[i+lookback, 0])  # Next day Close (feature 0)
    return np.array(X), np.array(y)

lookback = 60

# Create sequences
X_train, y_train = create_sequences(scaled_train, lookback)
X_val, y_val = create_sequences(scaled_val, lookback)
X_test, y_test = create_sequences(scaled_test, lookback)

print(f"Train sequences:")
print(f"  X: {X_train.shape}")
print(f"  y: {y_train.shape}")
print(f"\nVal sequences:")
print(f"  X: {X_val.shape}")
print(f"  y: {y_val.shape}")
print(f"\nTest sequences:")
print(f"  X: {X_test.shape}")
print(f"  y: {y_test.shape}")

## 6. Train/Validation/Test Split

In [ ]:
# Create PyTorch DataLoaders
batch_size = 32

train_dataset = TensorDataset(
    torch.FloatTensor(X_train),
    torch.FloatTensor(y_train.reshape(-1, 1))
)

val_dataset = TensorDataset(
    torch.FloatTensor(X_val),
    torch.FloatTensor(y_val.reshape(-1, 1))
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Inspect one batch
for X_batch, y_batch in train_loader:
    print(f"\nBatch shapes:")
    print(f"  X: {X_batch.shape}")
    print(f"  y: {y_batch.shape}")
    break

## 7. Build LSTM Model Architecture

In [ ]:
class LSTMPredictor(nn.Module):
    """Stacked LSTM for stock price prediction."""
    
    def __init__(self, input_size, hidden_size_1=128, hidden_size_2=64, dropout=0.2):
        super(LSTMPredictor, self).__init__()
        
        self.lstm1 = nn.LSTM(input_size, hidden_size_1, batch_first=True, dropout=dropout)
        self.dropout1 = nn.Dropout(dropout)
        
        self.lstm2 = nn.LSTM(hidden_size_1, hidden_size_2, batch_first=True, dropout=dropout)
        self.dropout2 = nn.Dropout(dropout)
        
        self.fc = nn.Linear(hidden_size_2, 1)
    
    def forward(self, x):
        # LSTM1: (batch, seq, input) -> (batch, seq, hidden1)
        lstm_out1, _ = self.lstm1(x)
        lstm_out1 = self.dropout1(lstm_out1)
        
        # LSTM2: (batch, seq, hidden1) -> (batch, seq, hidden2)
        lstm_out2, _ = self.lstm2(lstm_out1)
        lstm_out2 = self.dropout2(lstm_out2)
        
        # Take last timestep: (batch, seq, hidden2) -> (batch, hidden2)
        last_output = lstm_out2[:, -1, :]
        
        # FC layer: (batch, hidden2) -> (batch, 1)
        output = self.fc(last_output)
        return output

# Initialize model
device = 'cpu'
input_size = X_train.shape[2]  # number of features

model = LSTMPredictor(input_size=input_size).to(device)
print(model)

In [ ]:
# Count parameters
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total

total_params = count_parameters(model)
print(f"\nTotal trainable parameters: {total_params:,}")
print(f"Model size: ~{total_params * 4 / (1024**2):.2f} MB (assuming float32)")

## 8. Train the Model with Early Stopping

In [ ]:
# Training setup
optimizer = Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
num_epochs = 20  # Reduced for notebook
patience = 5
patience_counter = 0
best_val_loss = float('inf')

train_losses = []
val_losses = []

print("Starting training...\n")

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Patience: {patience_counter}/{patience}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping after {epoch+1} epochs")
        break

print("\nTraining completed!")

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train Loss', linewidth=2)
ax.plot(val_losses, label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Training History')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Evaluate Model Performance

In [ ]:
# Make predictions on test set
model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_pred_scaled = model(X_test_tensor).cpu().numpy()

# Inverse transform
def inverse_transform(scaled_predictions, scaler):
    """Inverse transform scaled predictions."""
    n_features = scaler.n_features_in_
    dummy = np.zeros((scaled_predictions.shape[0], n_features))
    dummy[:, 0] = scaled_predictions.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

y_pred = inverse_transform(y_pred_scaled, scaler)
y_test_unscaled = inverse_transform(y_test, scaler)

print(f"Predictions shape: {y_pred.shape}")
print(f"Test targets shape: {y_test_unscaled.shape}")

In [ ]:
# Calculate metrics
mae = np.mean(np.abs(y_test_unscaled - y_pred))
rmse = np.sqrt(np.mean((y_test_unscaled - y_pred) ** 2))
mape = np.mean(np.abs((y_test_unscaled - y_pred) / (np.abs(y_test_unscaled) + 1e-8))) * 100

# Directional accuracy
actual_direction = np.diff(y_test_unscaled)
pred_direction = np.diff(y_pred)
directional_acc = np.mean((actual_direction * pred_direction) > 0) * 100

# Print metrics
print("="*60)
print("EVALUATION METRICS")
print("="*60)
print(f"MAE:                   ${mae:.4f}")
print(f"RMSE:                  ${rmse:.4f}")
print(f"MAPE:                  {mape:.2f}%")
print(f"Directional Accuracy:  {directional_acc:.2f}%")
print("="*60)

## 10. Visualize Predictions vs Actual

In [ ]:
# Plot actual vs predicted
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y_test_unscaled, label='Actual', linewidth=2)
ax.plot(y_pred, label='Predicted', linewidth=2, alpha=0.8)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Close Price ($)')
ax.set_title(f'{ticker} Actual vs Predicted (MAPE: {mape:.2f}%)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Plot residuals
residuals = y_test_unscaled - y_pred

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(residuals, color='red', linewidth=1, alpha=0.7)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.fill_between(range(len(residuals)), residuals, 0, alpha=0.3, color='red')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Residual ($)')
ax.set_title(f'{ticker} Prediction Residuals')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"\nResidual Statistics:")
print(f"Mean: ${np.mean(residuals):.4f}")
print(f"Std: ${np.std(residuals):.4f}")
print(f"Min: ${np.min(residuals):.4f}")
print(f"Max: ${np.max(residuals):.4f}")

## 11. Calculate Trading Metrics (Sharpe Ratio)

In [ ]:
# Calculate Sharpe Ratio
# Trading strategy: buy if predicted > actual, else sell
signal = (y_pred > y_test_unscaled).astype(int) * 2 - 1  # -1 or 1

# Daily returns from strategy
actual_returns = np.diff(y_test_unscaled) / y_test_unscaled[:-1]
strategy_returns = signal[:-1] * actual_returns

mean_return = np.mean(strategy_returns)
std_return = np.std(strategy_returns) + 1e-8
sharpe = mean_return / std_return * np.sqrt(252)

print(f"\nTRADING STRATEGY METRICS")
print(f"="*60)
print(f"Mean Daily Return:     {mean_return*100:.4f}%")
print(f"Daily Std Dev:         {std_return*100:.4f}%")
print(f"Sharpe Ratio (252-day): {sharpe:.4f}")
print(f"="*60)
print(f"\nInterpretation:")
print(f"- Sharpe > 1: Good risk-adjusted return")
print(f"- Sharpe < 1: Return not compensating for risk")
print(f"- This model: {('promising' if sharpe > 0 else 'needs improvement')}")

In [ ]:
print("\n" + "="*60)
print("EDA NOTEBOOK COMPLETE")
print("="*60)
print("\nNext steps for production:")
print("1. Run: python src/train_main.py")
print("2. Start API: uvicorn api.main:app --reload")
print("3. Launch dashboard: streamlit run app/streamlit_app.py")
print("4. View MLflow: mlflow ui")
print("="*60)